# Construyendo el Backend para una Aplicación Full-Stack con CrewAI

## Plan Paso a Paso: Del Notebook al Backend con FastAPI

En el notebook `037-5th-top-use-case-crewai-2026.ipynb`, construimos un sistema de atención al cliente impulsado por IA usando **CrewAI Flows + Crews**. Ese notebook ejecuta todo dentro de Jupyter — escribes un ticket y la IA lo procesa allí mismo.

Pero, ¿y si queremos una **aplicación web real** donde los usuarios puedan enviar tickets desde un navegador? Para eso, necesitamos un **backend** — un servidor que reciba las peticiones del frontend y ejecute nuestra lógica de CrewAI.

**En este notebook, construiremos ese backend paso a paso.**

## ¿Qué es un Backend?

Un **backend** es un programa que se ejecuta en un servidor y espera peticiones. Cuando un usuario hace clic en "Enviar Ticket" en el navegador (el frontend), el frontend envía una petición al backend. El backend la procesa y devuelve una respuesta.

```
Navegador del Usuario (Frontend)       Servidor (Backend)
┌─────────────────────┐            ┌─────────────────────┐
│                     │  HTTP POST │                     │
│  Formulario de      │ ─────────> │  FastAPI recibe     │
│  Envío de Ticket    │            │  los datos del      │
│                     │            │  ticket             │
│                     │  JSON      │                     │
│  Mostrar Respuesta  │ <───────── │  CrewAI lo procesa  │
│                     │            │  y responde         │
└─────────────────────┘            └─────────────────────┘
```

Usaremos **FastAPI** como nuestro framework de backend — el mismo que usamos en la aplicación de tareas con LangChain (`050-todoapp-LC-backend`).

## La Visión General: Lo Que Vamos a Construir

Nuestro backend tendrá **un endpoint principal de API**:

| Método | Endpoint | Qué Hace |
|--------|----------|----------|
| POST | `/support/ticket` | Recibe un ticket de soporte, ejecuta CrewAI y devuelve la respuesta de la IA |

¡Eso es todo! Un solo endpoint. El Flow de CrewAI se encarga de toda la complejidad (clasificar, enrutar, resolver).

### Estructura del Proyecto

Siguiendo el mismo patrón que la aplicación de tareas con LangChain, la carpeta de nuestro backend tendrá esta estructura:

```
074-crewai-backend/
├── .env                  ← Claves de API (OpenAI)
├── main.py               ← Configuración de la app FastAPI + CORS
├── schemas.py            ← Modelos Pydantic (formato de petición/respuesta)
├── crew_logic.py         ← Todo el código de CrewAI (agentes, herramientas, flow)
└── routers/
    └── support.py        ← Endpoint de API para /support/ticket
```

**¿Por qué esta estructura?** Porque mantiene las cosas organizadas:
- `main.py` — Configura el servidor (mismo rol que en la app de tareas).
- `schemas.py` — Define qué datos entran y salen (mismo rol que en la app de tareas).
- `crew_logic.py` — Toda la lógica de CrewAI de nuestro notebook, empaquetada en una función reutilizable.
- `routers/support.py` — El endpoint de API que conecta el frontend con CrewAI (mismo patrón que `routers/todos.py` en la app de tareas).

Observa: **¡no necesitamos base de datos!** La app de tareas usaba PostgreSQL porque necesitaba almacenar tareas. Nuestra app solo procesa tickets y devuelve respuestas — no necesita almacenamiento.

## Paso 0: Instalar los Paquetes Necesarios

Necesitamos todo lo del notebook de CrewAI, además de **FastAPI** y **uvicorn** (el servidor que ejecuta FastAPI).

In [ ]:
# Instalar paquetes (ejecutar esto una sola vez)
#!pip install -U crewai python-dotenv fastapi uvicorn[standard]

## Paso 1: El Archivo `.env`

Igual que en la app de tareas, almacenamos nuestra clave de API en un archivo `.env`. El backend lee este archivo al arrancar.

Crea un archivo llamado `.env` dentro de la carpeta `074-crewai-backend/` con este contenido:

```
OPENAI_API_KEY=tu-clave-de-api-de-openai-aquí
```

**Importante:** Nunca subas tu archivo `.env` a git. Añádelo al `.gitignore`.

## Paso 2: `schemas.py` — Definir los Formatos de Petición y Respuesta

En la app de tareas, `schemas.py` definía `ToDoRequest` (lo que envía el frontend) y `ToDoResponse` (lo que devuelve el backend).

Hacemos lo mismo aquí. Cuando el frontend envía un ticket de soporte, nos manda estos campos:
- `customer_name`
- `customer_email`
- `ticket_subject`
- `ticket_message`

Y nosotros devolvemos la respuesta de la IA con:
- `category` — qué tipo de ticket es
- `complexity` — simple o complejo
- `urgency` — baja, media o alta
- `path_taken` — qué ruta siguió CrewAI
- `resolution_subject` — línea de asunto de la respuesta
- `resolution_message` — la respuesta completa de la IA
- `follow_up_needed` — si un humano debería hacer seguimiento

In [ ]:
%%writefile 074-crewai-backend/schemas.py
from pydantic import BaseModel


# Lo que el frontend nos envía
class TicketRequest(BaseModel):
    customer_name: str
    customer_email: str
    ticket_subject: str
    ticket_message: str


# Lo que nosotros devolvemos al frontend
class TicketResponse(BaseModel):
    customer_name: str
    customer_email: str
    category: str
    complexity: str
    urgency: str
    path_taken: str
    resolution_subject: str
    resolution_message: str
    follow_up_needed: bool

**Comparación con la app de tareas:**

| App de Tareas (`schemas.py`) | Nuestra App (`schemas.py`) |
|------------------------------|----------------------------|
| `ToDoRequest(name, completed)` | `TicketRequest(customer_name, email, subject, message)` |
| `ToDoResponse(name, completed, id)` | `TicketResponse(category, complexity, urgency, resolution_message, ...)` |

Mismo patrón, datos diferentes.

## Paso 3: `crew_logic.py` — El Cerebro de CrewAI

Este es el corazón de nuestro backend. Contiene **todo el código de CrewAI** del notebook `037-5th-top-use-case-crewai-2026.ipynb`, empaquetado en una única función llamada `process_ticket()`.

Piénsalo de esta manera:
- En el **notebook**, ejecutábamos el flow manualmente: creábamos un flow, establecíamos el estado, llamábamos a `kickoff()`.
- En el **backend**, la función `process_ticket()` hace todo eso automáticamente cuando el endpoint de la API la llama.

Esto es lo que contiene este archivo:
1. Los datos de FAQ y Base de Conocimiento (igual que en el notebook).
2. Las herramientas: `search_faq` y `search_knowledge_base` (igual que en el notebook).
3. Los agentes: Clasificador, Especialista en FAQ, Investigador, Redactor de Soluciones (igual que en el notebook).
4. Los modelos Pydantic para la salida de los agentes: `TicketClassification`, `SupportResponse` (igual que en el notebook).
5. El Estado del Flow: `SupportTicketState` (igual que en el notebook).
6. El Flow: `CustomerSupportFlow` (igual que en el notebook).
7. **NUEVO:** Una función `process_ticket()` que envuelve todo en una función fácil de llamar.

#### Nota de actualización: modelos actuales de OpenAI
* El ejemplo original es muy bueno para aprender el concepto, y funcionaba con el modelo de OpenAI con el que se grabó: `gpt-4o-mini`, con `temperature=0`.
* OpenAI va a retirar `gpt-4o-mini` (retirada prevista el 14 de enero de 2027). Con los modelos actuales, usaríamos `gpt-6-luna`, el modelo pequeño que recomienda OpenAI ahora. `MODEL` lo define una sola vez, para que puedas cambiarlo tocando una línea.
* Los modelos recientes de OpenAI solo aceptan la temperatura por defecto (1), así que el código usa `temperature=1`. La idea de `temperature=0` (respuestas más predecibles, útiles para clasificar los tickets siempre igual) sigue siendo importante, y otros modelos sí permiten elegirla.
* Los agentes de este flow usan herramientas. Por la vía que usa CrewAI, `gpt-6-luna` solo acepta herramientas con el razonamiento desactivado, así que el código añade `extra_body={"reasoning_effort": "none"}`. Así el modelo responde directamente, como hacía `gpt-4o-mini`. Funciona con la versión de CrewAI del repositorio del backend (1.9.3) y con la más reciente, la que instala `pip install -U crewai`.
* Consulta los modelos disponibles en developers.openai.com/api/docs/deprecations.

In [ ]:
%%writefile 074-crewai-backend/crew_logic.py
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from crewai.flow.flow import Flow, start, listen, router, or_

load_dotenv()

# ============================================================
# 1. BASES DE CONOCIMIENTO (igual que en el notebook)
# ============================================================

FAQ_DB = {
    "password_reset": {
        "question": "How do I reset my password?",
        "answer": (
            "To reset your password:\n"
            "1. Go to app.bigproject.com/login\n"
            "2. Click 'Forgot Password'\n"
            "3. Enter your email address\n"
            "4. Check your inbox for a reset link (arrives within 2 minutes)\n"
            "5. Click the link and set a new password\n\n"
            "If you don't receive the email, check your spam folder."
        ),
    },
    "pricing": {
        "question": "What are your pricing plans?",
        "answer": (
            "BigProject Pro pricing:\n"
            "- Free: Up to 5 users, basic features\n"
            "- Team: $15/user/month — unlimited projects, integrations\n"
            "- Enterprise: $45/user/month — SSO, audit logs, priority support\n\n"
            "All paid plans include a 14-day free trial. "
            "Visit bigproject.com/pricing for details."
        ),
    },
    "cancel_subscription": {
        "question": "How do I cancel my subscription?",
        "answer": (
            "To cancel your subscription:\n"
            "1. Go to Settings > Billing\n"
            "2. Click 'Manage Subscription'\n"
            "3. Select 'Cancel Plan'\n"
            "4. Confirm cancellation\n\n"
            "Your access continues until the end of the current billing period. "
            "No refunds for partial months."
        ),
    },
    "integrations": {
        "question": "What integrations do you support?",
        "answer": (
            "BigProject Pro integrates with 200+ tools including:\n"
            "- Slack, Microsoft Teams, Discord\n"
            "- GitHub, GitLab, Bitbucket\n"
            "- Jira, Linear, Asana\n"
            "- Google Drive, Dropbox, OneDrive\n\n"
            "Visit Settings > Integrations to connect your tools."
        ),
    },
}

KNOWLEDGE_BASE = {
    "billing": {
        "topic": "Billing & Refund Policies",
        "content": (
            "BILLING POLICIES:\n"
            "- Charges are processed on the 1st of each month.\n"
            "- Double charges can occur if a payment fails and retries.\n"
            "- Refund policy: Full refunds within 30 days of charge.\n"
            "- Partial refunds: Pro-rated for mid-cycle cancellations on annual plans.\n"
            "- To process a refund, a support agent must verify the charge in Stripe.\n"
            "- Refunds take 5-10 business days to appear on the customer's statement.\n"
            "- Escalate disputes over $500 to the billing manager."
        ),
    },
    "technical": {
        "topic": "Technical Troubleshooting",
        "content": (
            "COMMON TECHNICAL ISSUES:\n"
            "- API rate limits: 1000 requests/minute (Team), 5000/minute (Enterprise).\n"
            "- File upload limit: 100MB per file, 10GB total storage (Team).\n"
            "- 500 errors on upload: Usually caused by files exceeding size limit.\n"
            "- Sync issues: Clear browser cache, check internet connection.\n"
            "- SSO login failures: Verify SAML configuration in Settings > Security.\n"
            "- Webhook failures: Check endpoint URL and ensure HTTPS.\n"
            "- For persistent 500 errors, collect the request ID from the error page "
            "and escalate to the engineering team."
        ),
    },
    "account": {
        "topic": "Account Management",
        "content": (
            "ACCOUNT POLICIES:\n"
            "- Account deletion: Permanent, cannot be undone. 30-day grace period.\n"
            "- Data export: Available in Settings > Data > Export (CSV or JSON).\n"
            "- Team transfers: Admin can transfer ownership in Settings > Team.\n"
            "- Suspended accounts: Due to payment failure. Reactivate by updating payment.\n"
            "- GDPR data requests: Must be processed within 30 days."
        ),
    },
}


# ============================================================
# 2. HERRAMIENTAS (igual que en el notebook)
# ============================================================

@tool("search_faq")
def search_faq(query: str) -> str:
    """Search the FAQ database for answers to common customer questions.

    Use this tool to find pre-written answers for simple questions about
    password resets, pricing, cancellations, and integrations.

    Args:
        query: A keyword or topic to search for (e.g., 'password', 'pricing').
    """
    query_lower = query.lower()
    results = []
    for key, entry in FAQ_DB.items():
        if (
            query_lower in key
            or query_lower in entry["question"].lower()
            or query_lower in entry["answer"].lower()
        ):
            results.append(f"Q: {entry['question']}\nA: {entry['answer']}")
    if results:
        return "\n\n---\n\n".join(results)
    return f"No FAQ entries found for '{query}'. Try different keywords."


@tool("search_knowledge_base")
def search_knowledge_base(topic: str) -> str:
    """Search the detailed knowledge base for policies and procedures.

    Use this tool to find in-depth information about billing policies,
    technical troubleshooting steps, or account management procedures.

    Args:
        topic: The topic to search for (e.g., 'billing', 'technical', 'account').
    """
    topic_lower = topic.lower()
    results = []
    for key, entry in KNOWLEDGE_BASE.items():
        if (
            topic_lower in key
            or topic_lower in entry["topic"].lower()
            or topic_lower in entry["content"].lower()
        ):
            results.append(f"Topic: {entry['topic']}\n{entry['content']}")
    if results:
        return "\n\n---\n\n".join(results)
    return f"No knowledge base entries found for '{topic}'. Try different keywords."


# ============================================================
# 3. MODELOS DE SALIDA ESTRUCTURADA (igual que en el notebook)
# ============================================================

class TicketClassification(BaseModel):
    """Structured output from the Ticket Classifier."""
    category: str = Field(
        description=(
            "Ticket category: 'password_reset', 'pricing', "
            "'cancel_subscription', 'integrations', 'billing_issue', "
            "'technical_issue', or 'account_issue'"
        )
    )
    complexity: str = Field(
        description="'simple' (FAQ can answer) or 'complex' (needs investigation)"
    )
    urgency: str = Field(
        description="'low', 'medium', or 'high'"
    )
    reasoning: str = Field(
        description="Brief explanation of why this classification was chosen"
    )


class SupportResponse(BaseModel):
    """Structured response to send to the customer."""
    subject: str = Field(description="Response subject line")
    message: str = Field(description="The full response message to the customer")
    follow_up_needed: bool = Field(
        description="Whether a human agent should follow up"
    )


# ============================================================
# 4. AGENTES (igual que en el notebook)
# ============================================================

# Versión original: LLM(model="openai/gpt-4o-mini", temperature=0).
# OpenAI va a retirar gpt-4o-mini (retirada prevista el 14 de enero de 2027),
# así que usamos gpt-6-luna, el modelo pequeño que recomienda OpenAI ahora.
# Consulta los modelos disponibles en developers.openai.com/api/docs/deprecations
MODEL = "openai/gpt-6-luna"

# Los modelos recientes de OpenAI solo aceptan la temperatura por defecto (1),
# así que usamos 1 en lugar de 0. Los agentes de este flow usan herramientas y,
# por la vía que usa CrewAI (Chat Completions), gpt-6-luna solo acepta
# herramientas con el razonamiento desactivado, así que fijamos
# reasoning_effort en "none". Lo enviamos con extra_body, que funciona con la
# versión de CrewAI de este proyecto y con la más reciente.
llm = LLM(model=MODEL, temperature=1, extra_body={"reasoning_effort": "none"})

classifier_agent = Agent(
    role="Customer Support Ticket Classifier",
    goal=(
        "Accurately classify incoming support tickets by category "
        "and urgency level so they can be routed to the right team"
    ),
    backstory=(
        "You are a senior support operations specialist with 8 years "
        "of experience in customer support triage. You can quickly "
        "identify whether a ticket is a simple FAQ question or a "
        "complex issue requiring investigation."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

faq_agent = Agent(
    role="FAQ Support Specialist",
    goal=(
        "Quickly answer common customer questions using the FAQ "
        "database, providing clear and helpful responses"
    ),
    backstory=(
        "You are a friendly and efficient frontline support agent. "
        "You know the FAQ inside and out and can answer common "
        "questions in a warm, helpful tone."
    ),
    llm=llm,
    tools=[search_faq],
    verbose=True,
    allow_delegation=False,
)

research_agent = Agent(
    role="Support Research Specialist",
    goal=(
        "Thoroughly investigate complex customer issues by searching "
        "the knowledge base for relevant policies and procedures"
    ),
    backstory=(
        "You are a detail-oriented support investigator who leaves "
        "no stone unturned. When a customer has a billing dispute "
        "or technical problem, you dig into the knowledge base to "
        "find every relevant policy and procedure."
    ),
    llm=llm,
    tools=[search_knowledge_base],
    verbose=True,
    allow_delegation=False,
)

solution_agent = Agent(
    role="Customer Solution Specialist",
    goal=(
        "Write a clear, empathetic, and actionable response to the "
        "customer based on the research findings"
    ),
    backstory=(
        "You are a senior support agent known for turning frustrated "
        "customers into happy ones. You write responses that are "
        "empathetic, clear, and action-oriented."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)


# ============================================================
# 5. ESTADO DEL FLOW (igual que en el notebook)
# ============================================================

class SupportTicketState(BaseModel):
    """State that flows through the entire support pipeline."""
    customer_name: str = ""
    customer_email: str = ""
    ticket_subject: str = ""
    ticket_message: str = ""
    category: str = ""
    complexity: str = ""
    urgency: str = ""
    resolution_subject: str = ""
    resolution_message: str = ""
    follow_up_needed: bool = False
    path_taken: str = ""


# ============================================================
# 6. EL FLOW (igual que en el notebook)
# ============================================================

class CustomerSupportFlow(Flow[SupportTicketState]):
    """
    The AI-powered customer support flow.
    Classifies → Routes → Resolves → Formats.
    """

    @start()
    def classify_ticket(self):
        """Entry point: classify the incoming ticket."""
        print(f"\nClassifying ticket from {self.state.customer_name}...")

        classify_task = Task(
            description=(
                f"Classify this customer support ticket:\n"
                f"  Customer: {self.state.customer_name}\n"
                f"  Subject: {self.state.ticket_subject}\n"
                f"  Message: {self.state.ticket_message}\n\n"
                f"Determine:\n"
                f"1. Category: password_reset, pricing, cancel_subscription, "
                f"integrations, billing_issue, technical_issue, or account_issue\n"
                f"2. Complexity: 'simple' if it's a common FAQ question, "
                f"'complex' if it requires investigation\n"
                f"3. Urgency: 'low', 'medium', or 'high'"
            ),
            expected_output="A classification with category, complexity, urgency, and reasoning.",
            agent=classifier_agent,
            output_pydantic=TicketClassification,
        )

        classify_crew = Crew(
            agents=[classifier_agent],
            tasks=[classify_task],
            process=Process.sequential,
            verbose=True,
        )
        classify_crew.kickoff()

        if classify_task.output.pydantic:
            classification = classify_task.output.pydantic
            self.state.category = classification.category
            self.state.complexity = classification.complexity
            self.state.urgency = classification.urgency

        print(f"Classification: {self.state.category} | {self.state.complexity} | urgency={self.state.urgency}")
        return self.state.complexity

    @router(classify_ticket)
    def route_ticket(self):
        """Route to 'simple' or 'complex' path."""
        if self.state.complexity == "simple":
            print(">> Routing to: SIMPLE path (FAQ)")
            return "simple"
        else:
            print(">> Routing to: COMPLEX path (investigation)")
            return "complex"

    @listen("simple")
    def handle_simple_ticket(self):
        """Handle simple tickets with FAQ agent."""
        print("Simple path: FAQ Agent handling ticket...")

        faq_task = Task(
            description=(
                f"Answer this customer support question using the FAQ database:\n"
                f"  Customer: {self.state.customer_name}\n"
                f"  Question: {self.state.ticket_message}\n\n"
                f"Search the FAQ for relevant information and write a friendly, "
                f"helpful response. Address the customer by name."
            ),
            expected_output="A helpful response with clear instructions.",
            agent=faq_agent,
            output_pydantic=SupportResponse,
        )

        faq_crew = Crew(
            agents=[faq_agent],
            tasks=[faq_task],
            process=Process.sequential,
            verbose=True,
        )
        faq_crew.kickoff()

        if faq_task.output.pydantic:
            response = faq_task.output.pydantic
            self.state.resolution_subject = response.subject
            self.state.resolution_message = response.message
            self.state.follow_up_needed = response.follow_up_needed

        self.state.path_taken = "simple"
        return self.state.resolution_message

    @listen("complex")
    def handle_complex_ticket(self):
        """Handle complex tickets with investigation crew."""
        print("Complex path: Investigation Crew handling ticket...")

        research_task = Task(
            description=(
                f"Investigate this customer support issue:\n"
                f"  Customer: {self.state.customer_name}\n"
                f"  Category: {self.state.category}\n"
                f"  Issue: {self.state.ticket_message}\n\n"
                f"Search the knowledge base for relevant policies, procedures, "
                f"and troubleshooting steps."
            ),
            expected_output="A detailed research report with relevant policies and recommended actions.",
            agent=research_agent,
        )

        solution_task = Task(
            description=(
                f"Based on the research findings, write a response to "
                f"{self.state.customer_name} about their issue:\n"
                f"  Issue: {self.state.ticket_message}\n\n"
                f"Guidelines:\n"
                f"- Acknowledge their frustration\n"
                f"- Explain what happened (if applicable)\n"
                f"- Provide clear next steps\n"
                f"- Be empathetic and professional"
            ),
            expected_output="A professional, empathetic customer response with clear next steps.",
            agent=solution_agent,
            context=[research_task],
            output_pydantic=SupportResponse,
        )

        investigation_crew = Crew(
            agents=[research_agent, solution_agent],
            tasks=[research_task, solution_task],
            process=Process.sequential,
            verbose=True,
        )
        investigation_crew.kickoff()

        if solution_task.output.pydantic:
            response = solution_task.output.pydantic
            self.state.resolution_subject = response.subject
            self.state.resolution_message = response.message
            self.state.follow_up_needed = response.follow_up_needed

        self.state.path_taken = "complex"
        return self.state.resolution_message

    @listen(or_(handle_simple_ticket, handle_complex_ticket))
    def format_final_response(self, resolution):
        """Runs after either path completes. Returns the final state."""
        print(f"Formatting final response (path: {self.state.path_taken})")
        return self.state


# ============================================================
# 7. LA FUNCIÓN PRINCIPAL (NUEVO — esto es lo que llama la API)
# ============================================================

def process_ticket(
    customer_name: str,
    customer_email: str,
    ticket_subject: str,
    ticket_message: str,
) -> dict:
    """
    Process a support ticket through the CrewAI flow.

    This is the function that the API endpoint calls.
    It creates a flow, sets the ticket data, runs the flow,
    and returns the results as a dictionary.
    """
    # Create a new flow instance (fresh state for each ticket)
    flow = CustomerSupportFlow()

    # Set the ticket data on the flow state
    flow.state.customer_name = customer_name
    flow.state.customer_email = customer_email
    flow.state.ticket_subject = ticket_subject
    flow.state.ticket_message = ticket_message

    # Run the flow (this triggers the entire pipeline)
    flow.kickoff()

    # Return the results as a dictionary
    return {
        "customer_name": flow.state.customer_name,
        "customer_email": flow.state.customer_email,
        "category": flow.state.category,
        "complexity": flow.state.complexity,
        "urgency": flow.state.urgency,
        "path_taken": flow.state.path_taken,
        "resolution_subject": flow.state.resolution_subject,
        "resolution_message": flow.state.resolution_message,
        "follow_up_needed": flow.state.follow_up_needed,
    }

**La diferencia clave con el notebook:** La función `process_ticket()` al final. En el notebook, creábamos manualmente un flow y llamábamos a `kickoff()`. Aquí, envolvemos todo eso en una función para que el endpoint de la API pueda llamarla fácilmente.

Todo lo demás es idéntico al notebook.

## Paso 4: `routers/support.py` — El Endpoint de la API

Este archivo define el endpoint de la API al que llamará el frontend. Sigue el **mismo patrón exacto** que `routers/todos.py` en la app de tareas.

**Comparación:**

| App de Tareas (`routers/todos.py`) | Nuestra App (`routers/support.py`) |
|------------------------------------|------------------------------------|
| `@router.post("")` crea una tarea | `@router.post("/ticket")` procesa un ticket |
| Llama a `crud.create_todo(db, todo)` | Llama a `crew_logic.process_ticket(...)` |
| Devuelve la tarea guardada | Devuelve la respuesta de la IA |

Mismo patrón: recibir una petición → procesarla → devolver el resultado.

In [ ]:
%%writefile 074-crewai-backend/routers/support.py
from fastapi import APIRouter
import schemas
import crew_logic

router = APIRouter(
    prefix="/support"
)


@router.post("/ticket", response_model=schemas.TicketResponse)
async def process_support_ticket(ticket: schemas.TicketRequest):
    """
    Process a customer support ticket through the CrewAI flow.

    The frontend sends a ticket with:
      - customer_name, customer_email, ticket_subject, ticket_message

    The backend runs the CrewAI flow and returns:
      - category, complexity, urgency, resolution_message, etc.
    """
    result = crew_logic.process_ticket(
        customer_name=ticket.customer_name,
        customer_email=ticket.customer_email,
        ticket_subject=ticket.ticket_subject,
        ticket_message=ticket.ticket_message,
    )
    return result

**¡Eso es todo!** Solo 20 líneas de código. El endpoint:
1. Recibe un `TicketRequest` del frontend.
2. Llama a `crew_logic.process_ticket()` que ejecuta todo el Flow de CrewAI.
3. Devuelve el resultado como un `TicketResponse`.

Todo el trabajo pesado ocurre dentro de `crew_logic.py`.

## Paso 5: `main.py` — La Aplicación FastAPI

Este es el punto de entrada de nuestro backend. Crea la aplicación FastAPI, añade CORS (para que el frontend pueda comunicarse con el backend) e incluye nuestro router.

Es casi idéntico al `main.py` de la app de tareas.

In [ ]:
%%writefile 074-crewai-backend/main.py
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import PlainTextResponse
from starlette.exceptions import HTTPException as StarletteHTTPException

app = FastAPI()

# Include the support router
from routers import support
app.include_router(support.router)

# CORS configuration — allows the frontend (running on port 3000) to call our API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


# Global error handler
@app.exception_handler(StarletteHTTPException)
async def http_exception_handler(request, exc):
    return PlainTextResponse(str(exc.detail), status_code=exc.status_code)


# Root endpoint — just to check if the server is running
@app.get("/")
def read_root():
    return {"message": "CrewAI Support Backend is running!"}

**Comparación con el `main.py` de la app de tareas:**

| App de Tareas | Nuestra App |
|---------------|-------------|
| `from routers import todos` | `from routers import support` |
| `app.include_router(todos.router)` | `app.include_router(support.router)` |
| Usa `config.py` para configuración | Más simple — no necesita configuración de base de datos |
| CORS permite `localhost:3000` | Igual — CORS permite `localhost:3000` |

Casi idéntico. La única diferencia es que no necesitamos configuración de base de datos.

## Paso 6: Crear el Archivo `.env`

In [ ]:
%%writefile 074-crewai-backend/.env
OPENAI_API_KEY=your-openai-api-key-here

**Importante:** Reemplaza `your-openai-api-key-here` con tu clave de API de OpenAI real antes de ejecutar el servidor.

## Paso 7: Cómo Ejecutar el Backend

Para iniciar el servidor del backend, abre un terminal, navega a la carpeta `074-crewai-backend/` y ejecuta:

```bash
uvicorn main:app --reload
```

Esto inicia el servidor en `http://localhost:8000`.

- `main` se refiere a `main.py`.
- `app` se refiere al objeto `app = FastAPI()` dentro de `main.py`.
- `--reload` hace que el servidor se reinicie automáticamente cuando cambias el código (útil durante el desarrollo).

Deberías ver algo como:

```
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process
```

## Paso 8: Probar la API

Una vez que el servidor esté en marcha, puedes probarlo de tres formas:

### Opción A: Usar la documentación automática (la más fácil)

FastAPI genera automáticamente documentación interactiva. Abre tu navegador y ve a:

```
http://localhost:8000/docs
```

Verás el endpoint `/support/ticket`. Haz clic en él, luego haz clic en "Try it out", pega el JSON de abajo y haz clic en "Execute".

### Opción B: Usar `curl` desde el terminal

```bash
curl -X POST http://localhost:8000/support/ticket \
  -H "Content-Type: application/json" \
  -d '{
    "customer_name": "Alice Johnson",
    "customer_email": "alice@example.com",
    "ticket_subject": "No puedo iniciar sesión",
    "ticket_message": "Hola, he olvidado mi contraseña. ¿Cómo la restablezco?"
  }'
```

### Opción C: Usar Python (desde este notebook o cualquier script)

Asegúrate de que el servidor esté ejecutándose en un terminal primero, luego ejecuta la celda de abajo.

In [ ]:
import requests

# ¡Asegúrate de que el servidor esté ejecutándose primero! (uvicorn main:app --reload)
response = requests.post(
    "http://localhost:8000/support/ticket",
    json={
        "customer_name": "Alice Johnson",
        "customer_email": "alice@example.com",
        "ticket_subject": "Can't log in — need password reset",
        "ticket_message": "Hi, I forgot my password and I can't log into my account. How do I reset it? Thanks!",
    },
)

if response.ok:
    data = response.json()
    print(f"Categoría: {data['category']}")
    print(f"Complejidad: {data['complexity']}")
    print(f"Urgencia: {data['urgency']}")
    print(f"Ruta tomada: {data['path_taken']}")
    print(f"Seguimiento necesario: {data['follow_up_needed']}")
    print(f"\n--- Respuesta al cliente ---")
    print(f"Asunto: {data['resolution_subject']}")
    print(f"\n{data['resolution_message']}")
else:
    print(f"Error: {response.status_code}")
    print(response.text)

## Resumen: Del Notebook al Backend

Aquí tienes un resumen de lo que hicimos y cómo cada archivo se corresponde con el notebook original y la app de tareas:

### Cómo se Conectan los Archivos

```
Frontend (navegador)              Backend (servidor)
┌──────────────────┐             ┌─────────────────────────────────────┐
│                  │  POST       │  main.py                            │
│  Formulario de   │ ─────────>  │    └── routers/support.py           │
│  Envío de Ticket │  JSON       │          └── crew_logic.py          │
│                  │             │               ├── Herramientas      │
│                  │             │               ├── Agentes           │
│                  │  JSON       │               ├── Flow              │
│  Mostrar         │ <─────────  │               └── process_ticket()  │
│  Respuesta       │             │                                     │
└──────────────────┘             └─────────────────────────────────────┘
```

### Resumen Archivo por Archivo

| Archivo | Qué Hace | Inspirado En |
|---------|----------|--------------|
| `main.py` | Crea la app FastAPI, añade CORS, incluye el router | `050-todoapp-LC-backend/main.py` |
| `schemas.py` | Define los formatos de datos de petición/respuesta | `050-todoapp-LC-backend/schemas.py` |
| `routers/support.py` | Endpoint de API que recibe tickets | `050-todoapp-LC-backend/routers/todos.py` |
| `crew_logic.py` | Todo el código de CrewAI (agentes, herramientas, flow) | `037-5th-top-use-case-crewai-2026.ipynb` |
| `.env` | Almacenamiento de claves de API | `050-todoapp-LC-backend/.env` |

### Lo Que No Necesitamos (y Por Qué)

La app de tareas tenía varios archivos que **no necesitamos**:

| Archivo de la App de Tareas | Por Qué No Lo Necesitamos |
|-----------------------------|---------------------------|
| `database.py` | Sin base de datos — procesamos tickets sobre la marcha |
| `models.py` | Sin modelos de base de datos — no hay datos que almacenar |
| `crud.py` | Sin operaciones CRUD — no hay crear/leer/actualizar/eliminar |
| `config.py` | Sin configuración de base de datos — solo necesitamos la clave de API de OpenAI |
| `alembic/` | Sin migraciones de base de datos — no hay base de datos en absoluto |

Nuestro backend es más simple porque no almacena nada. Recibe un ticket, lo procesa con CrewAI y devuelve la respuesta.

## ¿Qué Viene Después?

Ahora que tenemos un backend funcionando, el siguiente paso es construir un **frontend** — una página web donde los usuarios puedan:
1. Rellenar un formulario con su nombre, email, asunto y mensaje.
2. Hacer clic en "Enviar Ticket".
3. Ver la respuesta de la IA mostrada en la página.

Ese frontend llamará a nuestro endpoint `POST /support/ticket`, igual que el frontend de la app de tareas llama a `POST /todos`.